# 模块一：SwellEx-96 纯 CPU 原始数据预处理与 Google Drive 持久化

本 Notebook 专为 **Google Colab 纯 CPU 运行时**（`Runtime -> Change runtime type -> CPU`）设计，完全不调用 PyTorch / CUDA，避免消耗宝贵的 Colab GPU 计算额度。

### 工作流职责与 Google Drive 目录约定
1. **自动定位原始数据（只读）**：默认指向 `MyDrive/Data/OpenAccess/SwellEx`（对应文件夹 [`1FfLqpZLlSP_R8mhTgwq5xvIUOJdC71Xj`](https://drive.google.com/drive/folders/1FfLqpZLlSP_R8mhTgwq5xvIUOJdC71Xj?usp=drive_link)）：
   - 同时兼容 **直接把 `Origindata` 内部四项（`environments/`, `events/`, `positions/`, `source/`）放在该目录**，或 **在该目录下保留 `Origindata/` 子目录** 两种结构；
   - 优先直读挂载路径 `/content/drive/MyDrive/Data/OpenAccess/SwellEx`，若路径调整则通过 `user.drive.id` / 快捷方式 ID / Drive API v3 自动兜底定位（规避 `gdown` 在 `ctds/` 52 文件目录上的 50 文件截断限制）；
   - 阵元通道数由 `positions/positions_vla.txt` 动态解析，声速剖面 `ctd_file` 仅在启用 `RBD` 时作为必需输入（纯 `SCM` 实验无需 CTD 文件），并将所需原始文件一次性缓存到实例本地高速 SSD（`/content/swellex_work/Origindata`）。
2. **本地高速盘按需特征提取**：支持通过 TOML 预设与参数覆盖配置 `[dataset]`、`[rbd]`、`[scm]`（含多快拍 `num_snapshots_per_segment`、重叠 `snapshot_overlap_count`、窗函数 `fft_window`、去均值 `signal_centering` 等），按需生成指定的 `METHODS`（至少指定 1 个：`["SCM"]`、`["RBD"]` 或 `["RBD", "SCM"]`），写入本地 SSD（`/content/swellex_work/outputs/`）。
3. **按参数组合独立落盘至 Google Drive**：每套参数组合拥有独立的 `CONFIG_KEY` 与各方法专属指纹目录（`by_method/<METHOD>/<dataset_name>/`），追加记录到 `dataset_manifest.json` 的 `configs` 表中，**不同参数组合相互独立、绝不互相覆盖**，供 **模块二（`02_network_training_gpu.ipynb`）** 精确消费。

> **注意**：在 Colab 内请直接按顺序运行下方单元格，**不要**调用 `./scripts/run.sh`（该脚本用于本地通过 SSH 派发到 `haiqin` 服务器）。

In [ ]:
# Cell 1: 环境检测、仓库引导、Google Drive 挂载与 Origindata (1FfLqpZLlSP_R8mhTgwq5xvIUOJdC71Xj) 自动适配
from collections import deque
from pathlib import Path
import importlib.util
import math
import os
import re
import shutil
import subprocess
import sys

# --- Google Drive 与仓库配置（对应 MyDrive/Data/{OpenAccess,SwellEx_Colab}） ---
ORIGINDATA_DRIVE_URL = "https://drive.google.com/drive/folders/1FfLqpZLlSP_R8mhTgwq5xvIUOJdC71Xj?usp=drive_link"
ORIGINDATA_DRIVE_SUBPATH = "Data/OpenAccess/SwellEx"  # MyDrive 下公开原始数据集路径
ORIGINDATA_LOCAL_OVERRIDE = os.environ.get("SWELLEX_ORIGINDATA_DIR", "").strip()  # 若需手动指定本地/挂载路径可填写
DRIVE_WORKSPACE_SUBDIR = "Data/SwellEx_Colab"  # 派生 HDF5 与实验输出保存在 MyDrive/Data/SwellEx_Colab/ 下
DEFAULT_EVENT_NAME = "S5"
DEFAULT_CTD_FILE = "i9605.prn"
REPO_GIT_URL = "https://github.com/AmazingUmi/SwellEx.git"
REPO_GIT_BRANCH = "main"
AUTO_UPDATE_REPO = True     # Colab 下每次运行 Cell 1 自动向 GitHub 拉取并同步最新提交
FORCE_RECLONE_REPO = False  # 若想彻底删除 /content/SwellEx 重新克隆，设为 True 运行一次即可
# ------------------------------------------------------------------

IN_COLAB = (
    "google.colab" in sys.modules
    or (importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None)
)


def extract_drive_folder_id(url_or_id: str) -> str:
    text = url_or_id.strip()
    for pattern in (r"/folders/([A-Za-z0-9_-]+)", r"[?&]id=([A-Za-z0-9_-]+)"):
        match = re.search(pattern, text)
        if match:
            return match.group(1)
    if re.fullmatch(r"[A-Za-z0-9_-]{15,}", text):
        return text
    raise ValueError(f"无法从输入解析 Google Drive Folder ID: {url_or_id!r}")


def count_vla_elements(positions_path: Path) -> int:
    """动态解析 positions_vla.txt 的有效阵元行数（至少 2 行有限数值）。"""
    if not positions_path.is_file():
        raise FileNotFoundError(f"缺少阵元位置文件: {positions_path}")
    rows = 0
    for raw_line in positions_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 2:
            raise ValueError(f"positions_vla.txt 行列数不足 2 列: {positions_path}")
        values = (float(parts[0]), float(parts[1]))
        if not all(math.isfinite(v) for v in values):
            raise ValueError(f"positions_vla.txt 含非有限数值: {positions_path}")
        rows += 1
    if rows < 2:
        raise ValueError(f"positions_vla.txt 至少需要 2 个有效阵元: {positions_path}")
    return rows


def origindata_rel_lists(
    base_dir: Path | None = None,
    *,
    event_name: str = DEFAULT_EVENT_NAME,
    ctd_file: str = DEFAULT_CTD_FILE,
    require_ctd: bool = False,
) -> tuple[tuple[str, ...], tuple[str, ...]]:
    """按阵元位置文件与方法需求返回 (必需相对路径, 可选相对路径)。"""
    num_elements = 21
    if base_dir is not None and (base_dir / "positions/positions_vla.txt").is_file():
        try:
            num_elements = count_vla_elements(base_dir / "positions/positions_vla.txt")
        except ValueError:
            num_elements = 21
    required = [
        "positions/positions_vla.txt",
        f"events/range/RangeEvent{event_name}/SproulToVLA.{event_name}.txt",
        *[f"events/{event_name}/vla_matfiles/{event_name}_VLA_NO_{idx}.mat" for idx in range(1, num_elements + 1)],
    ]
    optional = [
        f"source/Info_{event_name}.txt",
        f"source/{event_name}_SourceInfo.mat",
    ]
    ctd_rel = f"environments/ctds/{ctd_file}"
    if require_ctd:
        required.insert(1, ctd_rel)
    else:
        optional.insert(0, ctd_rel)
    return tuple(required), tuple(optional)


ORIGINDATA_FOLDER_ID = extract_drive_folder_id(ORIGINDATA_DRIVE_URL)

# 1. 检查是否误开了 GPU 运行时，防止白白消耗 Colab GPU 额度
has_nvidia = shutil.which("nvidia-smi") is not None and subprocess.run(
    ["nvidia-smi", "-L"], capture_output=True, text=True
).returncode == 0
if has_nvidia:
    print("⚠️ [额度提醒] 检测到当前实例挂载了 GPU！模块一仅使用纯 CPU（NumPy/SciPy/h5py），")
    print("   建议切换至 Runtime -> Change runtime type -> CPU 以节省 Colab GPU 计算单元。\n")
else:
    print("✅ 当前为纯 CPU 运行环境（零 GPU 额度消耗）。")

# 2. 先挂载 Google Drive（Colab 环境自动挂载；本地环境使用模拟/本地目录）
DRIVE_MOUNT_ROOT = Path("/content/drive")
if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=False)
    DRIVE_BASE = DRIVE_MOUNT_ROOT / "MyDrive" / DRIVE_WORKSPACE_SUBDIR
else:
    DRIVE_BASE = None  # 待 PROJECT_ROOT 解析后回退到 outputs/drive_staging


def get_git_commit_info(repo_dir: Path) -> tuple[str, str]:
    proc = subprocess.run(
        ["git", "-C", str(repo_dir), "log", "-1", "--format=%h|%s"],
        capture_output=True,
        text=True,
    )
    if proc.returncode == 0 and "|" in proc.stdout:
        short_sha, subject = proc.stdout.strip().split("|", 1)
        return short_sha, subject
    return "unknown", ""


def reload_project_modules() -> None:
    for mod_name in [m for m in list(sys.modules) if m == "scripts" or m.startswith("scripts.")]:
        sys.modules.pop(mod_name, None)
    importlib.invalidate_caches()


def ensure_project_root() -> Path:
    if not IN_COLAB:
        for candidate in (Path.cwd(), Path.cwd().parent):
            if (candidate / "scripts" / "remote_workflow" / "packs.py").exists():
                return candidate.resolve()
        raise FileNotFoundError("未找到项目根目录（缺少 scripts/remote_workflow/packs.py）。")

    colab_repo = Path("/content/SwellEx")
    if FORCE_RECLONE_REPO and colab_repo.exists():
        shutil.rmtree(colab_repo)
        print(f"🗑️ 已按 FORCE_RECLONE_REPO=True 清除旧目录: {colab_repo}")

    if (colab_repo / ".git").is_dir() and (colab_repo / "scripts" / "remote_workflow" / "packs.py").exists():
        old_sha, _ = get_git_commit_info(colab_repo)
        if AUTO_UPDATE_REPO:
            subprocess.run(["git", "-C", str(colab_repo), "remote", "set-url", "origin", REPO_GIT_URL], capture_output=True)
            fetch_proc = subprocess.run(
                ["git", "-C", str(colab_repo), "fetch", "--depth", "1", "origin", REPO_GIT_BRANCH],
                capture_output=True,
                text=True,
            )
            if fetch_proc.returncode == 0:
                subprocess.run(["git", "-C", str(colab_repo), "reset", "--hard", "FETCH_HEAD"], check=True, capture_output=True)
                subprocess.run(
                    ["git", "-C", str(colab_repo), "clean", "-fd", "-e", "outputs", "-e", "output", "-e", "Origindata"],
                    capture_output=True,
                )
                new_sha, subject = get_git_commit_info(colab_repo)
                if old_sha != new_sha:
                    print(f"🔄 已同步云端仓库最新提交: {old_sha} -> {new_sha} ({subject})")
                else:
                    print(f"✅ 云端仓库已是最新提交: {new_sha} ({subject})")
            else:
                print(f"⚠️ 检查远端更新失败（继续使用现有副本 {old_sha}）: {fetch_proc.stderr.strip()}")
        return colab_repo.resolve()

    if colab_repo.exists():
        shutil.rmtree(colab_repo)
    print(f"📥 正在克隆最新云端仓库 {REPO_GIT_URL} ({REPO_GIT_BRANCH}) -> {colab_repo} ...")
    proc = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_GIT_BRANCH, REPO_GIT_URL, str(colab_repo)],
        capture_output=True,
        text=True,
    )
    if proc.returncode == 0 and (colab_repo / "scripts" / "remote_workflow" / "packs.py").exists():
        new_sha, subject = get_git_commit_info(colab_repo)
        print(f"✅ 克隆完成: {new_sha} ({subject})")
        return colab_repo.resolve()

    fallback_candidates = [Path.cwd(), Path.cwd().parent]
    if DRIVE_MOUNT_ROOT.exists():
        fallback_candidates.extend([
            DRIVE_MOUNT_ROOT / "MyDrive" / "SwellEx",
            DRIVE_MOUNT_ROOT / "MyDrive" / DRIVE_WORKSPACE_SUBDIR / "SwellEx",
        ])
    for candidate in fallback_candidates:
        if (candidate / "scripts" / "remote_workflow" / "packs.py").exists():
            print(f"⚠️ GitHub 克隆未成功，已回退使用挂载目录副本: {candidate.resolve()}")
            return candidate.resolve()
    raise FileNotFoundError(
        f"自动克隆仓库失败（可能仓库为私有）: {proc.stderr.strip()}\n"
        "请先将最新 SwellEx 仓库上传至 /content/drive/MyDrive/SwellEx 或使用带权限链接克隆。"
    )


PROJECT_ROOT = ensure_project_root()
reload_project_modules()
REPO_COMMIT, REPO_SUBJECT = get_git_commit_info(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

if DRIVE_BASE is None:
    DRIVE_BASE = Path(os.environ.get("SWELLEX_DRIVE_SIM", PROJECT_ROOT / "outputs" / "drive_staging"))

# Colab 下统一使用实例本地 ext4 高速盘作为计算工作区（规避 Drive FUSE 不支持 symlink 与随机写慢）
WORK_ROOT = Path("/content/swellex_work") if IN_COLAB else PROJECT_ROOT
LOCAL_OUTPUTS = WORK_ROOT / "outputs"
if IN_COLAB:
    LOCAL_OUTPUTS.mkdir(parents=True, exist_ok=True)
DRIVE_DATASET_ROOT = DRIVE_BASE / "datasets"


def inspect_origindata_candidate(
    folder: Path,
    *,
    event_name: str = DEFAULT_EVENT_NAME,
    ctd_file: str = DEFAULT_CTD_FILE,
    require_ctd: bool = False,
) -> tuple[Path | None, list[str]]:
    """同时支持文件夹本身即 Origindata 内容，或文件夹内含 Origindata/ 子目录。"""
    default_req, _ = origindata_rel_lists(None, event_name=event_name, ctd_file=ctd_file, require_ctd=require_ctd)
    if not folder.exists():
        return None, list(default_req)
    partial_missing: list[str] = list(default_req)
    for cand in (folder, folder / "Origindata"):
        if not cand.is_dir():
            continue
        req_files, _ = origindata_rel_lists(cand, event_name=event_name, ctd_file=ctd_file, require_ctd=require_ctd)
        missing = [rel for rel in req_files if not (cand / rel).is_file()]
        if not missing:
            try:
                count_vla_elements(cand / "positions/positions_vla.txt")
            except ValueError as exc:
                missing = [f"positions/positions_vla.txt ({exc})"]
        if not missing:
            return cand, []
        if len(missing) < len(partial_missing):
            partial_missing = missing
    return None, partial_missing


def get_drivefs_id(path: Path) -> str | None:
    if not hasattr(os, "getxattr"):
        return None
    for attr in ("user.drive.id", "drive.id"):
        try:
            raw = os.getxattr(path, attr, follow_symlinks=False)
            return raw.decode("utf-8", errors="ignore").strip()
        except OSError:
            continue
    return None


def locate_origindata_on_mounted_drive(mount_root: Path, target_id: str) -> tuple[Path | None, str]:
    default_req, _ = origindata_rel_lists()
    direct_candidates = [
        (mount_root / "MyDrive" / ORIGINDATA_DRIVE_SUBPATH, "configured_drive_subpath"),
        (mount_root / "MyDrive" / "Data" / "open_access" / "SwellEx", "configured_drive_subpath"),
        (mount_root / ".shortcut-targets-by-id" / target_id, "drive_shortcut_id"),
        (mount_root / "MyDrive" / "Data" / "SwellEx", "mydrive_named_path"),
        (mount_root / "MyDrive" / "Origindata", "mydrive_named_path"),
        (mount_root / "MyDrive" / DRIVE_WORKSPACE_SUBDIR / "Origindata", "mydrive_named_path"),
    ]
    fallback_match: tuple[Path, str] | None = None
    best_partial: tuple[Path, list[str]] | None = None

    for cand_path, source_tag in direct_candidates:
        if not cand_path.exists():
            continue
        valid_dir, missing = inspect_origindata_candidate(cand_path)
        if valid_dir is not None:
            if (
                source_tag in ("configured_drive_subpath", "drive_shortcut_id")
                or get_drivefs_id(cand_path) == target_id
                or get_drivefs_id(valid_dir) == target_id
            ):
                return valid_dir, source_tag
            if fallback_match is None:
                fallback_match = (valid_dir, source_tag)
        elif len(missing) < len(default_req):
            best_partial = (cand_path, missing)

    skip_names = {
        ".Trash", ".ipynb_checkpoints", "__pycache__", ".git", "node_modules",
        "datasets", "experiments", "outputs", "output", "shipsear", "ShipsEar",
        "environments", "events", "positions", "source", "ctds", "vla_matfiles",
    }
    search_roots = [p for p in (mount_root / "MyDrive", mount_root / "Shareddrives") if p.is_dir()]
    queue: deque[tuple[Path, int]] = deque((root, 0) for root in search_roots)

    while queue:
        curr, depth = queue.popleft()
        if get_drivefs_id(curr) == target_id:
            valid_dir, missing = inspect_origindata_candidate(curr)
            if valid_dir is not None:
                return valid_dir, "drivefs_xattr_exact_id"
            best_partial = (curr, missing)
        elif fallback_match is None:
            if (curr / "positions" / "positions_vla.txt").is_file():
                valid_dir, missing = inspect_origindata_candidate(curr)
                if valid_dir is not None:
                    fallback_match = (valid_dir, "drivefs_structure_match")
                elif len(missing) < len(default_req):
                    best_partial = (curr, missing)

        if depth >= 5:
            continue
        try:
            with os.scandir(curr) as it:
                for entry in it:
                    if entry.name.startswith(".") or entry.name in skip_names:
                        continue
                    try:
                        if entry.is_dir(follow_symlinks=False):
                            queue.append((Path(entry.path), depth + 1))
                    except OSError:
                        continue
        except OSError:
            continue

    if fallback_match is not None:
        return fallback_match
    if best_partial is not None:
        folder_found, missing_list = best_partial
        preview = ", ".join(missing_list[:5])
        print(f"⚠️ 在 Google Drive 发现疑似目录 {folder_found}，但缺少 {len(missing_list)} 个必需文件（如: {preview}），请检查云盘同步是否完成。")
    return None, "not_found_on_mount"


def stage_origindata_to_local_ssd(
    src_origindata: Path,
    dst_origindata: Path,
    *,
    event_name: str = DEFAULT_EVENT_NAME,
    ctd_file: str = DEFAULT_CTD_FILE,
) -> Path:
    """将计算所需的核心原始文件（含存在时的 CTD 文件）复制到 Colab 本地 SSD。"""
    if src_origindata.resolve() == dst_origindata.resolve():
        return dst_origindata
    req_files, opt_files = origindata_rel_lists(
        src_origindata, event_name=event_name, ctd_file=ctd_file, require_ctd=False
    )
    copied = 0
    for rel in (*req_files, *opt_files):
        src_file = src_origindata / rel
        if not src_file.is_file():
            continue
        dst_file = dst_origindata / rel
        if dst_file.is_file() and dst_file.stat().st_size == src_file.stat().st_size:
            continue
        dst_file.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_file, dst_file)
        copied += 1
    if copied:
        print(f"⚡ 已将 {copied} 个原始数据文件从 Google Drive 缓存至本地高速盘: {dst_origindata}")
    return dst_origindata


def download_origindata_via_drive_api(
    target_id: str,
    dst_origindata: Path,
    *,
    event_name: str = DEFAULT_EVENT_NAME,
    ctd_file: str = DEFAULT_CTD_FILE,
    require_ctd: bool = False,
) -> tuple[Path | None, str]:
    """当文件夹为外部共享且未创建 MyDrive 快捷方式时，通过 Drive API v3 按需拉取原始文件。"""
    from google.colab import auth  # type: ignore
    from googleapiclient.discovery import build  # type: ignore
    from googleapiclient.http import MediaIoBaseDownload  # type: ignore

    print(f"🌐 正在通过 Google Drive API v3 解析文件夹 ID: {target_id} ...")
    auth.authenticate_user()
    service = build("drive", "v3", cache_discovery=False)

    if DRIVE_MOUNT_ROOT.is_dir():
        chain: list[str] = []
        curr_id: str | None = target_id
        seen: set[str] = set()
        while curr_id and curr_id not in seen:
            seen.add(curr_id)
            info = service.files().get(
                fileId=curr_id, fields="id, name, parents", supportsAllDrives=True
            ).execute()
            parents = info.get("parents") or []
            if not parents:
                break
            chain.append(info["name"])
            curr_id = parents[0]
        if chain:
            mounted_cand = DRIVE_MOUNT_ROOT / "MyDrive" / Path(*reversed(chain))
            valid_cand, _ = inspect_origindata_candidate(
                mounted_cand, event_name=event_name, ctd_file=ctd_file, require_ctd=require_ctd
            )
            if valid_cand is not None:
                stage_origindata_to_local_ssd(
                    valid_cand, dst_origindata, event_name=event_name, ctd_file=ctd_file
                )
                return dst_origindata, f"drive_api_parent_path:{valid_cand}"

    def list_children(folder_id: str) -> dict[str, dict]:
        items: dict[str, dict] = {}
        page_token = None
        while True:
            resp = service.files().list(
                q=f"'{folder_id}' in parents and trashed = false",
                fields="nextPageToken, files(id, name, mimeType, size)",
                pageSize=1000,
                pageToken=page_token,
                supportsAllDrives=True,
                includeItemsFromAllDrives=True,
            ).execute()
            for item in resp.get("files", []):
                items[item["name"]] = item
            page_token = resp.get("nextPageToken")
            if not page_token:
                break
        return items

    root_children = list_children(target_id)
    if "Origindata" in root_children and root_children["Origindata"]["mimeType"] == "application/vnd.google-apps.folder":
        root_children = list_children(root_children["Origindata"]["id"])

    folder_cache: dict[tuple[str, ...], dict[str, dict]] = {(): root_children}

    def resolve_remote_file(rel_posix: str) -> dict | None:
        parts = tuple(rel_posix.split("/"))
        curr_children = root_children
        for idx, dir_name in enumerate(parts[:-1]):
            prefix = parts[: idx + 1]
            if prefix not in folder_cache:
                dir_meta = curr_children.get(dir_name)
                if not dir_meta or dir_meta["mimeType"] != "application/vnd.google-apps.folder":
                    return None
                folder_cache[prefix] = list_children(dir_meta["id"])
            curr_children = folder_cache[prefix]
        return curr_children.get(parts[-1])

    def fetch_one(rel: str, *, required: bool) -> None:
        meta = resolve_remote_file(rel)
        if meta is None:
            if required:
                raise FileNotFoundError(f"Google Drive 文件夹 ({target_id}) 中缺少必需文件: {rel}")
            return
        dst_file = dst_origindata / rel
        remote_size = int(meta.get("size", -1))
        if dst_file.is_file() and (remote_size < 0 or dst_file.stat().st_size == remote_size):
            return
        dst_file.parent.mkdir(parents=True, exist_ok=True)
        request = service.files().get_media(fileId=meta["id"], supportsAllDrives=True)
        tmp_file = dst_file.with_suffix(dst_file.suffix + ".part")
        with tmp_file.open("wb") as fh:
            downloader = MediaIoBaseDownload(fh, request, chunksize=8 * 1024 * 1024)
            done = False
            while not done:
                _, done = downloader.next_chunk()
        tmp_file.replace(dst_file)

    # 先下载阵元位置文件以确定动态通道数，再拉取其余必需/可选文件
    fetch_one("positions/positions_vla.txt", required=True)
    req_files, opt_files = origindata_rel_lists(
        dst_origindata, event_name=event_name, ctd_file=ctd_file, require_ctd=require_ctd
    )
    for rel in req_files:
        fetch_one(rel, required=True)
    for rel in opt_files:
        fetch_one(rel, required=False)
    return dst_origindata, f"drive_api_v3:{target_id}"


# 3. 解析并准备 WORK_ROOT / "Origindata"
work_origindata = WORK_ROOT / "Origindata"
ORIGINDATA_RESOLUTION = "unresolved"
ORIGINDATA_SOURCE_PATH = ""

existing_local, _ = inspect_origindata_candidate(work_origindata)
if existing_local is not None and existing_local == work_origindata:
    ORIGINDATA_RESOLUTION = "local_ready"
    ORIGINDATA_SOURCE_PATH = str(work_origindata)
elif ORIGINDATA_LOCAL_OVERRIDE:
    override_valid, missing = inspect_origindata_candidate(Path(ORIGINDATA_LOCAL_OVERRIDE))
    if override_valid is None:
        raise FileNotFoundError(
            f"ORIGINDATA_LOCAL_OVERRIDE={ORIGINDATA_LOCAL_OVERRIDE!r} 缺少必需文件（共缺 {len(missing)} 个，如 {missing[:3]}）。"
        )
    ORIGINDATA_SOURCE_PATH = str(override_valid)
    if IN_COLAB:
        stage_origindata_to_local_ssd(override_valid, work_origindata)
    elif not work_origindata.exists():
        work_origindata.symlink_to(override_valid, target_is_directory=True)
    ORIGINDATA_RESOLUTION = "local_override"
elif IN_COLAB:
    mounted_src, mode = locate_origindata_on_mounted_drive(DRIVE_MOUNT_ROOT, ORIGINDATA_FOLDER_ID)
    if mounted_src is not None:
        ORIGINDATA_SOURCE_PATH = str(mounted_src)
        stage_origindata_to_local_ssd(mounted_src, work_origindata)
        ORIGINDATA_RESOLUTION = mode
    else:
        api_dst, mode = download_origindata_via_drive_api(ORIGINDATA_FOLDER_ID, work_origindata)
        ORIGINDATA_SOURCE_PATH = f"gdrive://{ORIGINDATA_FOLDER_ID}"
        ORIGINDATA_RESOLUTION = mode
else:
    repo_orig, _ = inspect_origindata_candidate(PROJECT_ROOT / "Origindata")
    if repo_orig is not None:
        ORIGINDATA_RESOLUTION = "local_repo"
        ORIGINDATA_SOURCE_PATH = str(repo_orig)

ORIGINDATA_READY = inspect_origindata_candidate(work_origindata)[0] == work_origindata

# 4. 检查数值依赖（Colab 默认自带 numpy / scipy / h5py）
REQUIRED_PKGS = ("numpy", "scipy", "h5py")
MISSING_PKGS = [pkg for pkg in REQUIRED_PKGS if importlib.util.find_spec(pkg) is None]

print(f"PROJECT_ROOT       : {PROJECT_ROOT} (commit={REPO_COMMIT})")
print(f"WORK_ROOT (本地盘) : {WORK_ROOT}")
print(f"Drive Folder ID    : {ORIGINDATA_FOLDER_ID}")
print(f"Origindata 就绪    : {ORIGINDATA_READY} (来源={ORIGINDATA_RESOLUTION}, 路径={ORIGINDATA_SOURCE_PATH or work_origindata})")
print(f"DRIVE_DATASET_ROOT : {DRIVE_DATASET_ROOT}")
if MISSING_PKGS:
    print(f"ℹ️ 当前环境缺少数值包 {MISSING_PKGS}（后续单元将以配置校验/干跑模式执行；在 Colab 可运行 `!pip install -q h5py numpy scipy`）。")
else:
    print("✅ 数值依赖 (numpy, scipy, h5py) 已全部就绪。")

## 1. 选择实验预设、按需特征路由与多快拍/信号处理参数

下方参数单元直接复用仓库内置的官方 TOML 配置（包含完整的 `[dataset]`、`[rbd]`、`[scm]` 表），并经过 `generation_config.py` 严格模式校验：
- `PRESET`（或 `CUSTOM_CONFIG_TOML`）：
  - `"train_before"`：主实验预设（`range_overlap`，CPA 前训练、后测试，中央留白 `gap_s`）
  - `"train_after"`：反向泛化预设（`range_overlap`，CPA 后训练、前测试）
  - `"full"`：整段交错划分预设（`full_interleaved`，仅允许无跨窗样本重叠配置）
- `METHODS`：**必须显式指定且至少包含 1 个方法**（如仅跑多快拍协方差矩阵设 `["SCM"]`，或双路对照设 `["RBD", "SCM"]`）。仅选 `["SCM"]` 时不读取 CTD 声速剖面。
- `DATASET_OVERRIDES` / `RBD_OVERRIDES` / `SCM_OVERRIDES`：可在不改仓库预设文件的前提下覆盖任意 TOML 字段（例如多快拍 `SCM_OVERRIDES = {"num_snapshots_per_segment": 4, "snapshot_overlap_count": 2}`，当组合快拍步长 `(Ns - overlap) * segment_step_s > test_step_s` 时配合 `DATASET_OVERRIDES = {"fft_window": "hann", "signal_centering": "per_window", "test_step_s": 0.0}` 自动继承特征窗口步长）。
- **各参数组合完全独立**：系统对归一化后的参数组合计算唯一哈希并生成 `CONFIG_KEY`，每一套配置均独立生成和保存各自的结果，绝不覆盖其他参数组合。
- `QUICK_SMOKE_MODE`：默认 `False`；设为 `True` 时对 `range_overlap` 自动截取 CPA（约 `3541 s`）附近 `3200..3900 s` 窗口（兼容重叠多快拍 SCM），对 `full_interleaved` 截取前 `300 s`，方便几十秒内跑通端到端联调。

In [ ]:
# Cell 2: 生成任务参数配置（每套参数组合自动生成独立 CONFIG_KEY）
from copy import deepcopy
import hashlib
import json
import tomllib

from scripts.data_processing.generation_config import (
    normalize_methods,
    validate_generation_tables,
)
from scripts.remote_workflow.config import load_config

# --- 用户可调配置 ---
PRESET = "train_before"            # 可选: "train_before" | "train_after" | "full"
CUSTOM_CONFIG_TOML = None          # 若指定自定义 TOML 路径（str/Path），优先于 PRESET
METHODS = ["RBD", "SCM"]           # 必须指定，数目 >= 1，例如 ["SCM"] 或 ["RBD", "SCM"]

# 按需覆盖 TOML 中 [dataset] / [rbd] / [scm] 的任意参数（留空 {} 则完全使用 TOML 值）：
# 多快拍 SCM 示例（test_step_s=0.0 表示测试集自动继承组合后的 SCM 窗口步长）：
#   METHODS = ["SCM"]
#   DATASET_OVERRIDES = {"fft_window": "hann", "signal_centering": "per_window", "test_step_s": 0.0}
#   SCM_OVERRIDES = {"num_snapshots_per_segment": 4, "snapshot_overlap_count": 2}
DATASET_OVERRIDES: dict = {}
RBD_OVERRIDES: dict = {}
SCM_OVERRIDES: dict = {}

QUICK_SMOKE_MODE = False           # False=完整数据；True=截取小窗口快速冒烟联调（兼容多快拍重叠 SCM）
FORCE_REGENERATE = False           # True=强制删除本地同指纹目录重新计算；False=复用已完好通过的结果
# --------------------

PRESET_TOML_MAP = {
    "train_before": PROJECT_ROOT / "scripts/remote_workflow/presets/train_before.toml",
    "train_after": PROJECT_ROOT / "scripts/remote_workflow/presets/train_after.toml",
    "full": PROJECT_ROOT / "scripts/remote_workflow/presets/full.toml",
}

if CUSTOM_CONFIG_TOML is not None:
    config_toml_path = Path(CUSTOM_CONFIG_TOML).expanduser().resolve()
    config_base_name = config_toml_path.stem
else:
    if PRESET not in PRESET_TOML_MAP:
        raise ValueError(f"未知 PRESET={PRESET!r}，可选: {list(PRESET_TOML_MAP)}")
    config_toml_path = PRESET_TOML_MAP[PRESET]
    config_base_name = PRESET

if not isinstance(METHODS, (list, tuple)) or len(METHODS) < 1:
    raise ValueError("METHODS 必须显式指定为非空列表/元组（数目最小为 1），如 ['SCM'] 或 ['RBD', 'SCM']。")
ACTIVE_METHODS = normalize_methods(list(METHODS), allow_none=False)


def merge_section_overrides(base: dict, overrides: dict, section_name: str) -> dict:
    merged = deepcopy(base)
    for key, val in overrides.items():
        if key not in merged:
            raise ValueError(f"[{section_name}] 存在未知覆盖字段: {key!r}（合法字段: {sorted(merged)}）")
        if isinstance(merged[key], dict) and isinstance(val, dict):
            nested = deepcopy(merged[key])
            for n_key, n_val in val.items():
                if n_key not in nested:
                    raise ValueError(f"[{section_name}.{key}] 存在未知覆盖字段: {n_key!r}")
                nested[n_key] = deepcopy(n_val)
            merged[key] = nested
        else:
            merged[key] = deepcopy(val)
    return merged


validated_base_cfg = load_config(config_toml_path)
dataset_cfg = merge_section_overrides(validated_base_cfg["dataset"], DATASET_OVERRIDES, "dataset")
rbd_cfg = merge_section_overrides(validated_base_cfg["rbd"], RBD_OVERRIDES, "rbd")
scm_cfg = merge_section_overrides(validated_base_cfg["scm"], SCM_OVERRIDES, "scm")

if QUICK_SMOKE_MODE:
    if dataset_cfg["split_strategy"] == "range_overlap":
        # 围绕 S5 最近距离点（CPA ~ 3541s）截取双侧短窗口，兼容多快拍重叠 SCM
        dataset_cfg["segment_start_s"] = 3200.0
        dataset_cfg["segment_end_s"] = 3900.0
        dataset_cfg["half_duration_s"] = min(float(dataset_cfg["half_duration_s"]), 200.0)
    else:
        start_s = 0.0 if float(dataset_cfg["segment_start_s"]) < 0 else float(dataset_cfg["segment_start_s"])
        dataset_cfg["segment_end_s"] = start_s + 299.0
    existing_tag = dataset_cfg["manual_dataset_variant_tag"].strip()
    dataset_cfg["manual_dataset_variant_tag"] = f"{existing_tag}_colab_smoke" if existing_tag else "colab_smoke"

# 使用官方共享 schema 校验全部生成表（含跨切分原始采样点防泄漏检查）
norm_dataset_preview, norm_rbd_preview, norm_scm_preview = validate_generation_tables(
    deepcopy(dataset_cfg),
    deepcopy(rbd_cfg),
    deepcopy(scm_cfg),
    methods=ACTIVE_METHODS,
)

config_identity_payload = {
    "schema_version": 1,
    "methods": list(ACTIVE_METHODS),
    "dataset": (
        norm_dataset_preview
        if "RBD" in ACTIVE_METHODS
        else {k: v for k, v in norm_dataset_preview.items() if k != "ctd_file"}
    ),
    "rbd": norm_rbd_preview if "RBD" in ACTIVE_METHODS else None,
    "scm": norm_scm_preview if "SCM" in ACTIVE_METHODS else None,
}
CONFIG_SHA256 = hashlib.sha256(
    json.dumps(config_identity_payload, sort_keys=True, separators=(",", ":"), allow_nan=False).encode("utf-8")
).hexdigest()

methods_tag = "+".join(ACTIVE_METHODS)
scm_tag = (
    f"_ns{norm_scm_preview['num_snapshots_per_segment']}ov{norm_scm_preview['snapshot_overlap_count']}"
    if "SCM" in ACTIVE_METHODS
    and (norm_scm_preview["num_snapshots_per_segment"] != 1 or norm_scm_preview["snapshot_overlap_count"] != 0)
    else ""
)
smoke_tag = "_smoke" if QUICK_SMOKE_MODE else ""
CONFIG_KEY = f"{config_base_name}_{methods_tag}{scm_tag}{smoke_tag}_{CONFIG_SHA256[:8]}"

GENERATION_PAYLOAD = {
    "schema_version": 1,
    "dataset": dataset_cfg,
    "rbd": rbd_cfg,
    "scm": scm_cfg,
    "result_path": f"outputs/generation_report_{CONFIG_KEY}.json",
}

print(f"已加载基础 TOML : {config_toml_path}")
print(f"  独立 CONFIG_KEY: {CONFIG_KEY}")
print(f"  启用特征路由   : {list(ACTIVE_METHODS)}")
print(f"  切分策略       : {norm_dataset_preview['split_strategy']} (train_side={norm_dataset_preview['train_side']}, gap_s={norm_dataset_preview['gap_s']}s)")
print(f"  窗口与信号预处理: start={norm_dataset_preview['segment_start_s']}s, end={norm_dataset_preview['segment_end_s']}s, "
      f"dur={norm_dataset_preview['segment_duration_s']}s, step={norm_dataset_preview['segment_step_s']}s, "
      f"fft_window={norm_dataset_preview['fft_window']}, centering={norm_dataset_preview['signal_centering']}")
if "SCM" in ACTIVE_METHODS:
    print(f"  SCM 快拍配置   : Ns={norm_scm_preview['num_snapshots_per_segment']}, "
          f"overlap={norm_scm_preview['snapshot_overlap_count']}, norm_floor={norm_scm_preview['norm_floor']}")
if "RBD" in ACTIVE_METHODS:
    print(f"  RBD 波束配置   : beam={norm_rbd_preview['beam_selection']}, est={norm_rbd_preview['frequency_estimation']}, "
          f"angles=[{norm_rbd_preview['angle_min_deg']}, {norm_rbd_preview['angle_max_deg']}] ({norm_rbd_preview['num_angles']} pts), "
          f"ctd={norm_dataset_preview['ctd_file']}")

## 2. 按需生成特征（`METHODS`）并执行 HDF5 契约校验

所有写入均在实例本地高速盘 `<WORK_ROOT>/outputs/Datasets/` 完成：
- 每个方法目录按其专属计算指纹 `generation_identity(norm_cfg, method)` 独立命名与复用（互不干扰）；
- 自动执行单文件结构校验（`inspect_h5`）与训练/测试切分无泄漏校验（`split_check`）；
- 当且仅当同时启用 `RBD` 与 `SCM` 且两者共享相同单快拍窗口与频点网格时，额外执行跨路由对齐校验（`crossroute_check`）。

In [ ]:
# Cell 3: 按需提取指定 METHODS 的 HDF5 特征并执行结构/切分契约校验
import json

GENERATED_LOCAL: dict[str, dict] = {}


def ensure_origindata_for_active_config(norm_cfg: dict, methods: tuple[str, ...]) -> tuple[bool, list[str]]:
    """按当前配置的 event_name / ctd_file / RBD 是否启用，校验或按需补充本地 Origindata 文件。"""
    event_name = norm_cfg["dataset"]["event_name"]
    ctd_file = norm_cfg["dataset"]["ctd_file"]
    require_ctd = "RBD" in methods
    valid_local, missing = inspect_origindata_candidate(
        work_origindata, event_name=event_name, ctd_file=ctd_file, require_ctd=require_ctd
    )
    if valid_local == work_origindata:
        return True, []
    if IN_COLAB and ORIGINDATA_SOURCE_PATH:
        if ORIGINDATA_SOURCE_PATH.startswith("gdrive://"):
            download_origindata_via_drive_api(
                ORIGINDATA_FOLDER_ID,
                work_origindata,
                event_name=event_name,
                ctd_file=ctd_file,
                require_ctd=require_ctd,
            )
        elif Path(ORIGINDATA_SOURCE_PATH).is_dir():
            stage_origindata_to_local_ssd(
                Path(ORIGINDATA_SOURCE_PATH),
                work_origindata,
                event_name=event_name,
                ctd_file=ctd_file,
            )
        valid_local, missing = inspect_origindata_candidate(
            work_origindata, event_name=event_name, ctd_file=ctd_file, require_ctd=require_ctd
        )
    return valid_local == work_origindata, missing


def load_or_generate_routes(
    norm_cfg: dict,
    *,
    methods: tuple[str, ...],
    method_hashes: dict[str, str],
    generate_fn,
    work_root: Path,
    force: bool,
) -> dict:
    report_file = Path(norm_cfg["result_path"])
    if not force and report_file.is_file():
        cached = json.loads(report_file.read_text(encoding="utf-8"))
        dirs_ok = (
            cached.get("status") == "PASS"
            and cached.get("methods") == list(methods)
            and all(
                Path(cached.get("datasets", {}).get(m, "")).is_dir()
                and Path(cached["datasets"][m]).name.endswith(f"_{method_hashes[m]}")
                and any(Path(cached["datasets"][m]).glob("*_train.h5"))
                and any(Path(cached["datasets"][m]).glob("*_test.h5"))
                for m in methods
            )
        )
        if dirs_ok:
            print(f"♻️ 复用本地已完成的特征目录 (CONFIG_KEY={CONFIG_KEY}, hashes={method_hashes})")
            return cached

    datasets_parent = work_root / "outputs" / "Datasets"
    if datasets_parent.is_dir():
        for method in methods:
            for stale in datasets_parent.glob(f"*_{method_hashes[method]}"):
                if stale.is_dir():
                    shutil.rmtree(stale)
    return generate_fn(norm_cfg, root=work_root, methods=methods)


if MISSING_PKGS:
    print(f"⏭️ 跳过实际特征计算（当前环境缺少 {MISSING_PKGS}）。Cell 2 配置 schema 校验已通过。")
else:
    from scripts.data_processing.dataset_validation import (
        crossroute_check,
        inspect_h5,
        split_check,
    )
    from scripts.data_processing.generation_config import normalize_config
    from scripts.data_processing.generation_core import (
        generate_datasets,
        generation_identity,
        json_value,
    )
    from scripts.data_processing.reporting import normalize_expected

    norm_cfg = normalize_config(GENERATION_PAYLOAD, root=WORK_ROOT, methods=ACTIVE_METHODS)
    ready_for_cfg, missing_for_cfg = ensure_origindata_for_active_config(norm_cfg, ACTIVE_METHODS)
    if not ready_for_cfg:
        print(
            f"⏭️ 跳过实际特征计算：Origindata 未就绪（当前配置缺少 {len(missing_for_cfg)} 个文件，示例: {missing_for_cfg[:3]}）。"
        )
    else:
        method_hashes = {m: generation_identity(norm_cfg, m) for m in ACTIVE_METHODS}
        print(f"🚀 开始准备特征数据集: CONFIG_KEY={CONFIG_KEY}, method_hashes={method_hashes} ...")
        gen_result = load_or_generate_routes(
            norm_cfg,
            methods=ACTIVE_METHODS,
            method_hashes=method_hashes,
            generate_fn=generate_datasets,
            work_root=WORK_ROOT,
            force=FORCE_REGENERATE,
        )
        route_dirs = {method: Path(gen_result["datasets"][method]) for method in ACTIVE_METHODS}

        expected_cfg = normalize_expected(norm_cfg, methods=ACTIVE_METHODS)
        all_records = []
        for method, folder in route_dirs.items():
            method_records = []
            for h5_path in sorted(p for p in folder.glob("*.h5") if not p.name.startswith("._")):
                rec, _ = inspect_h5(h5_path, method, expected_cfg)
                method_records.append(rec)
                all_records.append(rec)
            split_check(method_records, expected_cfg)

        same_samples = (
            set(ACTIVE_METHODS) == {"RBD", "SCM"}
            and norm_cfg["scm"]["num_snapshots_per_segment"] == 1
            and norm_cfg["scm"]["snapshot_overlap_count"] == 0
            and not (
                norm_cfg["rbd"]["frequency_estimation"] == "full"
                and norm_cfg["dataset"]["frequency_selection_modes"] != ["full"]
            )
        )
        if same_samples:
            crossroute_mode = "aligned_dual_routes"
            crossroute_check(all_records)
        else:
            crossroute_mode = "independent_routes"

        per_method_summary = {}
        for method in ACTIVE_METHODS:
            meta = gen_result["dataset_results"][method]
            per_method_summary[method] = {
                "dataset_name": meta["dataset_name"],
                "dataset_dir": str(route_dirs[method]),
                "method_hash": method_hashes[method],
                "train_count": int(meta["train_count"]),
                "test_count": int(meta["test_count"]),
                "num_freq_bins": len(meta["frequency_hz"]),
                "effective_segment_start_s": float(meta["effective_segment_start_s"]),
                "effective_segment_end_s": float(meta["effective_segment_end_s"]),
            }

        GENERATED_LOCAL[CONFIG_KEY] = {
            "config_key": CONFIG_KEY,
            "preset": config_base_name,
            "config_sha256": CONFIG_SHA256,
            "methods": list(ACTIVE_METHODS),
            "crossroute_validation": crossroute_mode,
            "generation_config": json_value({
                "schema_version": norm_cfg["schema_version"],
                "dataset": norm_cfg["dataset"],
                "rbd": norm_cfg["rbd"],
                "scm": norm_cfg["scm"],
            }),
            "routes": per_method_summary,
        }
        print(f"✅ HDF5 生成与契约校验全部通过 (crossroute={crossroute_mode}):")
        for method, info in per_method_summary.items():
            rel_folder = Path(info["dataset_dir"]).relative_to(WORK_ROOT)
            print(
                f"   - {method:4s}: {rel_folder} | "
                f"train={info['train_count']}, test={info['test_count']}, freq_bins={info['num_freq_bins']}"
            )

## 3. 按独立配置同步 HDF5 到 Google Drive 并更新 `dataset_manifest.json`

- 各方法的 HDF5 目录按自身不可变名称落盘至 `DRIVE_DATASET_ROOT/by_method/<METHOD>/<dataset_name>/`；
- `dataset_manifest.json` 在 `configs[CONFIG_KEY]` 下独立记录每一套参数组合的完整配置与路由索引，**保留历史全部参数组合，互不覆盖**。
- 同步完成后即可**关闭并释放当前 CPU 运行时**，切换到 **模块二（GPU）** 使用打印出的 `CONFIG_KEY` 启动训练。

In [ ]:
# Cell 4: 将当前参数组合生成的 HDF5 独立同步到 Google Drive 并写入 dataset_manifest.json
from datetime import datetime, timezone
import hashlib
import json

FORCE_RESYNC_DRIVE = False         # False=云端同指纹目录文件齐全且大小一致时跳过重复上传；True=强制重新复制
MANIFEST_PATH = DRIVE_DATASET_ROOT / "dataset_manifest.json"


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def sync_method_dir_to_drive(src_dir: Path, method: str, *, force_resync: bool) -> dict:
    rel_dest = Path("by_method") / method / src_dir.name
    dst_dir = DRIVE_DATASET_ROOT / rel_dest
    src_files = sorted(p for p in src_dir.iterdir() if p.is_file() and not p.name.startswith("._"))
    already_synced = (
        not force_resync
        and dst_dir.is_dir()
        and all((dst_dir / p.name).is_file() and (dst_dir / p.name).stat().st_size == p.stat().st_size for p in src_files)
    )
    if not already_synced:
        if dst_dir.exists():
            shutil.rmtree(dst_dir)
        dst_dir.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src_dir, dst_dir)

    h5_files = {}
    for h5_file in sorted(p for p in dst_dir.glob("*.h5") if not p.name.startswith("._")):
        h5_files[h5_file.name] = {
            "size_mb": round(h5_file.stat().st_size / (1024 * 1024), 3),
            "sha256": sha256_file(h5_file),
        }
    return {
        "rel_dir": rel_dest.as_posix(),
        "reused_existing_drive_copy": already_synced,
        "files": h5_files,
    }


if CONFIG_KEY not in GENERATED_LOCAL:
    print("ℹ️ 本次未产生新的本地 HDF5 目录，展示 Google Drive 独立配置目标结构示例:")
    example_layout = {
        "origindata_drive_folder_id": ORIGINDATA_FOLDER_ID,
        "drive_dataset_root": str(DRIVE_DATASET_ROOT),
        "manifest_file": str(MANIFEST_PATH),
        "config_key": CONFIG_KEY,
        "entries": {
            f"by_method/{m}/<split>_<variant>_<method_hash12>": "*_train.h5, *_test.h5, *_metadata.json"
            for m in ACTIVE_METHODS
        },
    }
    print(json.dumps(example_layout, ensure_ascii=False, indent=2))
else:
    DRIVE_DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    manifest = (
        json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
        if MANIFEST_PATH.exists()
        else {"schema_version": 2, "configs": {}}
    )
    manifest["schema_version"] = 2
    manifest.setdefault("configs", {})
    manifest["origindata_source"] = {
        "drive_folder_id": ORIGINDATA_FOLDER_ID,
        "drive_url": ORIGINDATA_DRIVE_URL,
        "resolution_mode": ORIGINDATA_RESOLUTION,
        "source_path": ORIGINDATA_SOURCE_PATH,
    }
    manifest["repo_commit"] = REPO_COMMIT

    local_bundle = GENERATED_LOCAL[CONFIG_KEY]
    synced_routes = {}
    for method, route_info in local_bundle["routes"].items():
        sync_meta = sync_method_dir_to_drive(
            Path(route_info["dataset_dir"]),
            method,
            force_resync=FORCE_RESYNC_DRIVE or FORCE_REGENERATE,
        )
        synced_routes[method] = {
            "dataset_name": route_info["dataset_name"],
            "method_hash": route_info["method_hash"],
            "train_count": route_info["train_count"],
            "test_count": route_info["test_count"],
            "num_freq_bins": route_info["num_freq_bins"],
            **sync_meta,
        }

    manifest["configs"][CONFIG_KEY] = {
        "config_key": CONFIG_KEY,
        "preset": local_bundle["preset"],
        "config_sha256": local_bundle["config_sha256"],
        "methods": local_bundle["methods"],
        "crossroute_validation": local_bundle["crossroute_validation"],
        "updated_at_utc": datetime.now(timezone.utc).isoformat(),
        "repo_commit": REPO_COMMIT,
        "generation_config": local_bundle["generation_config"],
        "routes": synced_routes,
    }
    manifest["latest_config_key"] = CONFIG_KEY

    MANIFEST_PATH.write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
    )
    print(f"🎉 已同步至 Google Drive 并独立记录配置: {MANIFEST_PATH}")
    print(f"🔑 本次数据集 CONFIG_KEY = {CONFIG_KEY!r}（当前清单共保存 {len(manifest['configs'])} 套独立配置）")
    print("👉 现在可断开 CPU 运行时，在 notebooks/02_network_training_gpu.ipynb 中指定该 CONFIG_KEY 启动训练！")